In [2]:
import sys
sys.path.append("..")
sys.path.append("../scripts")

In [11]:
from ms2maccs import MS2Data, MACCSModel, collate_fn
from utils import calc_tanimoto, evaluate
from matchms.importing import load_from_mgf

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import ConcatDataset

import optuna
from optuna.samplers import TPESampler

In [14]:
MGF_TRAIN = "../ms2_data/train_specs_H_p_n_mode.mgf"
MGF_VAL_P = "../ms2_data/val_specs_H_p_mode.mgf"
MGF_VAL_N = "../ms2_data/val_specs_H_n_mode.mgf"
FP_BIT_MAP_P_MODE_PATH = "../fp_bit_maps/fp_bit_map_H_p_mode.pkl"
FP_BIT_MAP_N_MODE_PATH = "../fp_bit_maps/fp_bit_map_H_n_mode.pkl"

DEVICE = "cuda"
N_TRIALS = 5

In [6]:
ms2train = MS2Data(MGF_TRAIN, FP_BIT_MAP_P_MODE_PATH, FP_BIT_MAP_N_MODE_PATH)
ms2val_p = MS2Data(MGF_VAL_P, FP_BIT_MAP_P_MODE_PATH, FP_BIT_MAP_N_MODE_PATH)
#ms2val_n = MS2Data(MGF_VAL_N, FP_BIT_MAP_P_MODE_PATH, FP_BIT_MAP_N_MODE_PATH)
#ms2val = ConcatDataset([ms2val_p, ms2val_n])

Processing train_specs_H_p_n_mode.mgf: 27170it [01:24, 319.68it/s]
Processing val_specs_H_p_mode.mgf: 4409it [00:12, 343.06it/s]


In [20]:
def train(model, train_loader, val_loader, lr, weight_decay, n_epochs, trial):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

    best_tanimoto = -1.0
    best_state = None # maybe tanimoto is not the best indication alone

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0

        for submaccs, maccs in train_loader:
            maccs = maccs.to(DEVICE)
            optimizer.zero_grad()
            logits = model(submaccs)
            loss = loss_fn(logits, maccs)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(maccs)

        train_loss = epoch_loss / len(train_loader.dataset)
        val_loss, val_tanimoto = evaluate(model, val_loader, loss_fn)
        scheduler.step(val_tanimoto)

        print(f"{epoch:>6}  {train_loss:>11.4f}  {val_loss:>9.4f}  {val_tanimoto:>13.4f}")

        if val_tanimoto > best_tanimoto:
            best_tanimoto = val_tanimoto
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        
        trial.report(val_tanimoto, epoch)
        if trial.should_prune():
            raise optuna.exception.TrialPruned()

    print(f"\nBest validation Tanimoto: {best_tanimoto:.4f}")
    model.load_state_dict(best_state)
    
    return model, best_tanimoto

In [21]:
def make_loaders(batch_size):
    ms2train_loader = DataLoader(ms2train, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
    ms2val_loader = DataLoader(ms2val_p, batch_size=batch_size, collate_fn=collate_fn)

    return ms2train_loader, ms2val_loader

In [22]:
def objective(trial):
    d_model = trial.suggest_categorical("d_model", [128, 256, 512, 1024])
    nhead = trial.suggest_categorical("nhead", [2 ,4 ,8 ,16 ,32])
    num_layers = trial.suggest_int("num_layers", 2, 8)
    dropout = trial.suggest_float("dropout", 0.0, 0.4, step=0.05)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16, 32, 64])

    train_loader, val_loader = make_loaders(batch_size)

    model = MACCSModel(d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout).to(DEVICE)

    best_tanimoto, _ = train(model, 
                             train_loader, 
                             val_loader, 
                             lr=lr, 
                             weight_decay=weight_decay, 
                             n_epochs=10, 
                             trial=trial)

    return best_tanimoto

In [23]:
sampler = TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="ms2maccs_opt",
)

study.enqueue_trial({
    "d_model": 512,
    "nhead": 8,
    "num_layers": 5,
    "dropout": 0.15,
    "lr": 3e-4,
    "weight_decay": 1e-3,
    "batch_size": 16,
})

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial

[I 2026-05-29 14:27:41,212] A new study created in memory with name: ms2maccs_opt


  0%|          | 0/5 [00:00<?, ?it/s]

     1       0.3036     0.3133         0.4967
     2       0.2725     0.2982         0.5166
     3       0.2602     0.3034         0.5183
[W 2026-05-29 14:31:16,257] Trial 0 failed with parameters: {'d_model': 512, 'nhead': 8, 'num_layers': 5, 'dropout': 0.15, 'lr': 0.0003, 'weight_decay': 0.001, 'batch_size': 16} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\dietr004\AppData\Local\anaconda3\envs\nts\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\dietr004\AppData\Local\Temp\ipykernel_23072\2242115312.py", line 14, in objective
    best_tanimoto, _ = train(model,
                      ^^^^^^^^^^^^
  File "C:\Users\dietr004\AppData\Local\Temp\ipykernel_23072\2517909983.py", line 21, in train
    epoch_loss += loss.item() * len(maccs)
                  ^^^^^^^^^^^
KeyboardInterrupt
[W 2026-05-29 14:31:16,259] Trial 0 failed 

KeyboardInterrupt: 